In [1]:
import os
print(os.getcwd())

/workspace


In [1]:
import os
print(os.getcwd())

/workspace/observer-effect-mech-interp


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer loaded.")
print("Tokenizer length:", len(tokenizer))
print("Tokenizer vocab_size:", tokenizer.vocab_size)

Tokenizer loaded.
Tokenizer length: 151669
Tokenizer vocab_size: 151643


In [5]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="cuda"
)

model.eval()

print("Model loaded.")
print("Model class:", type(model))
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)
print("Model vocab_size:", model.config.vocab_size)
print("Number of layers:", model.config.num_hidden_layers)
print("Hidden size:", model.config.hidden_size)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded.
Model class: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
Model device: cuda:0
Model dtype: torch.bfloat16
Model vocab_size: 151936
Number of layers: 36
Hidden size: 2560


In [6]:
TEXT = "The Eiffel Tower is in"

inputs = tokenizer(TEXT, return_tensors="pt")
inputs = {k: v.to("cuda") for k, v in inputs.items()}

print("Text:", TEXT)
print("Input IDs shape:", inputs["input_ids"].shape)
print("Input IDs:", inputs["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

Text: The Eiffel Tower is in
Input IDs shape: torch.Size([1, 7])
Input IDs: tensor([[  785,   468,  3092,   301, 21938,   374,   304]], device='cuda:0')
Tokens: ['The', 'ĠE', 'iff', 'el', 'ĠTower', 'Ġis', 'Ġin']


In [8]:
with torch.no_grad():
    outputs = model(**inputs)

In [9]:
print("Logits shape:", outputs.logits.shape)

Logits shape: torch.Size([1, 7, 151936])


In [10]:
last_logits = outputs.logits[0, -1, :]

print("Last-position logits shape:", last_logits.shape)

Last-position logits shape: torch.Size([151936])


In [11]:
top_values, top_indices = torch.topk(last_logits, k=5)

for rank, (token_id, logit) in enumerate(
    zip(top_indices.tolist(), top_values.tolist()),
    start=1
):
    token = tokenizer.decode([token_id])
    print(f"{rank}. token={repr(token):15} id={token_id:<8} logit={logit:.4f}")

1. token=' Paris'        id=12095    logit=21.1250
2. token=' the'          id=279      logit=18.7500
3. token=' France'       id=9625     logit=18.3750
4. token=' which'        id=892      logit=17.8750
5. token=' what'         id=1128     logit=17.0000


In [12]:
probs = torch.softmax(last_logits.float(), dim=-1)

top_probs, top_indices = torch.topk(probs, k=5)

for rank, (token_id, prob) in enumerate(
    zip(top_indices.tolist(), top_probs.tolist()),
    start=1
):
    token = tokenizer.decode([token_id])
    print(f"{rank}. token={repr(token):15} probability={prob:.4%}")

1. token=' Paris'        probability=79.7778%
2. token=' the'          probability=7.4205%
3. token=' France'       probability=5.1000%
4. token=' which'        probability=3.0933%
5. token=' what'         probability=1.2895%


In [13]:
probs = torch.softmax(last_logits.float(), dim=-1)

top_probs, top_indices = torch.topk(probs, k=5)

for rank, (token_id, prob) in enumerate(
    zip(top_indices.tolist(), top_probs.tolist()),
    start=1
):
    token = tokenizer.decode([token_id])
    print(f"{rank}. token={repr(token):15} probability={prob:.4%}")

print("\nProbability sum:", probs.sum().item())

1. token=' Paris'        probability=79.7778%
2. token=' the'          probability=7.4205%
3. token=' France'       probability=5.1000%
4. token=' which'        probability=3.0933%
5. token=' what'         probability=1.2895%

Probability sum: 1.0000003576278687


In [14]:
probs = torch.softmax(last_logits.float(), dim=-1)

top_probs, top_indices = torch.topk(probs, k=5)

for rank, (token_id, prob) in enumerate(
    zip(top_indices.tolist(), top_probs.tolist()),
    start=1
):
    token = tokenizer.decode([token_id])
    print(f"{rank}. token={repr(token):15} probability={prob:.4%}")

print("\nProbability sum:", probs.sum().item())

1. token=' Paris'        probability=79.7778%
2. token=' the'          probability=7.4205%
3. token=' France'       probability=5.1000%
4. token=' which'        probability=3.0933%
5. token=' what'         probability=1.2895%

Probability sum: 1.0000003576278687


In [15]:
print(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [16]:
print("Number of layers:", len(model.model.layers))
print("Layer 0 type:", type(model.model.layers[0]))
print("Layer 18 type:", type(model.model.layers[18]))
print("Layer 35 type:", type(model.model.layers[35]))

Number of layers: 36
Layer 0 type: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3DecoderLayer'>
Layer 18 type: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3DecoderLayer'>
Layer 35 type: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3DecoderLayer'>


In [17]:
captured = {}

def capture_layer_output(module, inputs, output):
    hidden = output[0] if isinstance(output, tuple) else output
    captured["layer18"] = hidden.detach()

handle = model.model.layers[18].register_forward_hook(
    capture_layer_output
)

with torch.no_grad():
    hooked_outputs = model(**inputs)

handle.remove()

activation = captured["layer18"]

print("Captured activation shape:", activation.shape)
print("Activation device:", activation.device)
print("Activation dtype:", activation.dtype)

Captured activation shape: torch.Size([1, 7, 2560])
Activation device: cuda:0
Activation dtype: torch.bfloat16


In [18]:
last_token_activation = activation[0, -1, :]

print("Last-token activation shape:", last_token_activation.shape)
print("Last-token activation dtype:", last_token_activation.dtype)
print("Last-token activation device:", last_token_activation.device)

print("First 10 values:", last_token_activation[:10])
print("L2 norm:", torch.linalg.vector_norm(last_token_activation.float()).item())

Last-token activation shape: torch.Size([2560])
Last-token activation dtype: torch.bfloat16
Last-token activation device: cuda:0
First 10 values: tensor([17.5000, -3.8438, -1.8281,  9.9375, 14.5625,  2.2656,  0.9688, -2.2969,
         2.7188,  1.1172], device='cuda:0', dtype=torch.bfloat16)
L2 norm: 50.405704498291016


In [19]:
original_logits = outputs.logits
hooked_logits = hooked_outputs.logits

max_abs_diff = (
    original_logits.float() - hooked_logits.float()
).abs().max().item()

same_logits = torch.allclose(
    original_logits.float(),
    hooked_logits.float(),
    atol=1e-5,
    rtol=1e-5
)

original_top_token = original_logits[0, -1, :].argmax().item()
hooked_top_token = hooked_logits[0, -1, :].argmax().item()

print("Original logits shape:", original_logits.shape)
print("Hooked logits shape:", hooked_logits.shape)
print("Max absolute difference:", max_abs_diff)
print("Logits allclose:", same_logits)

print("Original top token:",
      repr(tokenizer.decode([original_top_token])))

print("Hooked top token:",
      repr(tokenizer.decode([hooked_top_token])))

Original logits shape: torch.Size([1, 7, 151936])
Hooked logits shape: torch.Size([1, 7, 151936])
Max absolute difference: 0.0
Logits allclose: True
Original top token: ' Paris'
Hooked top token: ' Paris'
